In [2]:
import os
import sys

if "SUMO_HOME" in os.environ:
    tools = os.path.join(os.environ["SUMO_HOME"], "tools")
    sys.path.append(tools)
    print("SUMO_HOME trovato:", os.environ["SUMO_HOME"])
else:
    raise EnvironmentError(
        "Variabile SUMO_HOME non impostata. "
        "Verifica la configurazione fatta in precedenza su Windows."
    )

import traci
import gymnasium as gym
from sumo_rl import SumoEnvironment

from pathlib import Path
from stable_baselines3.dqn.dqn import DQN
from stable_baselines3.ppo.ppo import PPO

print("Import completati correttamente.")

SUMO_HOME trovato: C:\Program Files (x86)\Eclipse\Sumo\
Import completati correttamente.


In [3]:
net_dir = Path("sumo_rl/nets/nostri")
net_file = net_dir / "cross.net.xml"
route_file = net_dir / "cross_flows.rou.xml"

print("Cerco in:", net_dir.resolve())
print("net_file esiste:", net_file.exists())
print("route_file esiste:", route_file.exists())

Cerco in: C:\VScode\sumo-rl-project\sumo-rl-pedestrians\sumo_rl\nets\nostri
net_file esiste: True
route_file esiste: True


In [6]:
env = SumoEnvironment(
    net_file=str(net_file),
    route_file=str(route_file),
    out_csv_name="outputs/2way-single-intersection/baseline_dqn",
    single_agent=True,
    use_gui=False,
    num_seconds=20000,
)

model = DQN(
    env=env,
    policy="MlpPolicy",
    learning_rate=0.001,
    learning_starts=0,
    train_freq=1,
    target_update_interval=500,
    exploration_initial_eps=0.05,
    exploration_final_eps=0.01,
    verbose=1,
)

print("Modello creato.")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Modello creato.


In [8]:
model.learn(total_timesteps=8000, progress_bar=True)

model.save("outputs/2way-single-intersection/dqn_baseline_model")
print("Training completato e modello salvato.")

Output()

Training completato e modello salvato.


In [19]:
import pandas as pd
import glob

files = sorted(glob.glob("outputs/2way-single-intersection/baseline_dqn_conn1_ep*.csv"))
print(f"File trovati: {len(files)}")

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print(f"Step totali: {len(df)}")
print(f"\n--- Veicoli ---")
print(f"Waiting time medio:  {df['system_mean_waiting_time'].mean():.2f}s")
print(f"Waiting time finale: {df['system_mean_waiting_time'].iloc[-1]:.2f}s")

print(f"\n--- Pedoni ---")
print(f"Waiting time medio:  {df['system_mean_waiting_time_pedestrians'].mean():.2f}s")
print(f"Waiting time finale: {df['system_mean_waiting_time_pedestrians'].iloc[-1]:.2f}s")

print(f"\n--- Colonne disponibili ---")
print(df.columns.tolist())

File trovati: 3
Step totali: 8083

--- Veicoli ---
Waiting time medio:  3571.33s
Waiting time finale: 0.00s

--- Pedoni ---
Waiting time medio:  5.29s
Waiting time finale: 0.00s

--- Colonne disponibili ---
['step', 'system_total_running', 'system_total_backlogged', 'system_total_stopped', 'system_total_arrived', 'system_total_departed', 'system_total_teleported', 'system_total_waiting_time', 'system_mean_waiting_time', 'system_mean_speed', 'system_total_running_pedestrians', 'system_total_stopped_pedestrians', 'system_total_arrived_pedestrians', 'system_total_departed_pedestrians', 'system_total_waiting_time_pedestrians', 'system_mean_waiting_time_pedestrians', 'system_mean_speed_pedestrians', 'J3_stopped', 'J3_accumulated_waiting_time', 'J3_average_speed', 'agents_total_stopped', 'agents_total_accumulated_waiting_time']


In [20]:
for f in files:
    d = pd.read_csv(f)
    print(f"{f.split('/')[-1]:40s}  teleportati={d['system_total_teleported'].iloc[-1]:.0f}  veh_wait={d['system_mean_waiting_time'].mean():.1f}s")

2way-single-intersection\baseline_dqn_conn1_ep1.csv  teleportati=0  veh_wait=29.6s
2way-single-intersection\baseline_dqn_conn1_ep2.csv  teleportati=0  veh_wait=6884.3s
2way-single-intersection\baseline_dqn_conn1_ep3.csv  teleportati=0  veh_wait=330.1s


In [ ]:
import pandas as pd

df = pd.read_csv(r".\sumo-rl-pedestrians\outputs\2way-single-intersection\baseline_dqn_conn1_ep1.csv")

print(f"Step totali: {len(df)}")
print(f"\n--- Veicoli ---")
print(f"Waiting time medio:  {df['system_mean_waiting_time'].mean():.2f}s")
print(f"Waiting time finale: {df['system_mean_waiting_time'].iloc[-1]:.2f}s")

print(f"\n--- Pedoni ---")
print(f"Waiting time medio:  {df['system_mean_waiting_time_pedestrians'].mean():.2f}s")
print(f"Waiting time finale: {df['system_mean_waiting_time_pedestrians'].iloc[-1]:.2f}s")

print(f"\n--- Colonne disponibili ---")
print(df.columns.tolist())

Step totali: 81

--- Veicoli ---
Waiting time medio:  29.59s
Waiting time finale: 35.65s

--- Pedoni ---
Waiting time medio:  5.24s
Waiting time finale: 0.58s

--- Colonne disponibili ---
['step', 'system_total_running', 'system_total_backlogged', 'system_total_stopped', 'system_total_arrived', 'system_total_departed', 'system_total_teleported', 'system_total_waiting_time', 'system_mean_waiting_time', 'system_mean_speed', 'system_total_running_pedestrians', 'system_total_stopped_pedestrians', 'system_total_arrived_pedestrians', 'system_total_departed_pedestrians', 'system_total_waiting_time_pedestrians', 'system_mean_waiting_time_pedestrians', 'system_mean_speed_pedestrians', 'J3_stopped', 'J3_accumulated_waiting_time', 'J3_average_speed', 'agents_total_stopped', 'agents_total_accumulated_waiting_time']


In [12]:
model = PPO(
    env=env,
    policy="MlpPolicy",
    verbose=1,
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [13]:
model.learn(total_timesteps=200, progress_bar=True)

model.save("outputs/2way-single-intersection/dqn_baseline_model")
print("Training completato e modello salvato.")

Output()

-----------------------------
| time/              |      |
|    fps             | 29   |
|    iterations      | 1    |
|    time_elapsed    | 69   |
|    total_timesteps | 2048 |
-----------------------------


Training completato e modello salvato.
